# MySQL Loader


In [ ]:
import os
import pandas as pd
import pymysql

def load_data_from_mysql(sample_size=20000, target_attribute="Smiling", include_metadata=False):
    """
    Load facial attribute data directly from the unified MySQL master view (celeba_master_view).
    Joins images_metadata, celebrities, celeba_identities, and celeba_attributes in a single query.
    Falls back gracefully to list_attr_celeba.csv if MySQL is unavailable.
    """
    try:
        conn = pymysql.connect(
            host="127.0.0.1", port=3306, user="root", password="",
            database="celeba_db", connect_timeout=3
        )
        query = f"""
            SELECT 
                file_path,
                `{target_attribute}` AS label,
                celebrity_name,
                identity_cluster,
                gender,
                age_group
            FROM celeba_master_view
            WHERE `{target_attribute}` IS NOT NULL
            LIMIT {sample_size};
        """
        df = pd.read_sql(query, conn)
        conn.close()
        print(f"Loaded {len(df)} rows from MySQL celeba_master_view.")
    except Exception:
        print("MySQL offline or master view unavailable, using local CSV fallback.")
        df = pd.read_csv("data/metadata/list_attr_celeba.csv", nrows=sample_size)
        df["file_path"] = df["image_id"].apply(
            lambda name: os.path.join("img_align_celeba", name.replace(".jpg", ".png"))
        )
        df["label"] = df[target_attribute]
        df["celebrity_name"] = "Unknown"
        df["identity_cluster"] = 0
        df["gender"] = df["Male"].apply(lambda x: "Male" if x == 1 else "Female")
        df["age_group"] = df["Young"].apply(lambda x: "Young" if x == 1 else "Senior")

    df["label"] = (df["label"] == 1).astype(int)
    if include_metadata:
        return df[["file_path", "label", "celebrity_name", "identity_cluster", "gender", "age_group"]].copy()
    return df[["file_path", "label"]].copy()
